In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsmoothie.smoother import GaussianSmoother
import spikeinterface
import spikeinterface.full as si

import helper_functions as helper
from spikeinterface.sorters import run_sorter_local

In [ ]:
local_path= '/mnt/disk20tb/Organoids/Maxone/IMR90_T3_03262024_AP/IMR90_T3_03262024_AP/240326/16719/Network/000011/data.raw.h5' #network data from chip 16848
number_of_wells = 1
def get_recording_objects(local_path):
    recording_objects = []
    
    for i in range(number_of_wells):
        well_id = f'well{i:03d}'
        stream_id = f'well{i:03d}'

        recording = si.read_maxwell(local_path, stream_id=stream_id)
       
        
        channel_ids = recording.get_channel_ids()
        fs = recording.get_sampling_frequency()
        num_chan = recording.get_num_channels()
        num_seg = recording.get_num_segments()
        total_recording = recording.get_total_duration()

        print(f'Well ID: {well_id}')
        print('Sampling frequency:', fs)
        print('Number of channels:', num_chan)
        print('Number of segments:', num_seg)
        print(f"Total recording: {total_recording} s")
        print()
        recording_bp = si.bandpass_filter(recording, freq_min=300, freq_max=3000)

        recodring_cmr = si.common_reference(recording_bp, reference='global', operator='median')
        #recording_chunk = recodring_cmr.frame_slice(start_frame= 1*fs,end_frame=425*fs)
        recording_chunk = recodring_cmr.frame_slice(start_frame= 0*fs,end_frame=300*fs)
        #recording_chunk =si.scale(recording_chunk, gain=3.0)
        print(f"chunk duration: {recording_chunk.get_total_duration()} s")
        recording_objects.append(recording_chunk)
    return recording_objects

recording_objects = get_recording_objects(local_path)


In [ ]:
#BEFORE PLOTS
fig, axs = plt.subplots(len(recording_objects), figsize=(10, 6))

for i, recording in enumerate(recording_objects):
    locations = recording.get_channel_locations()
    axs[i].set_facecolor('#000000')
    for x, y in locations:
        axs[i].scatter(x, y, s=5, c='deepskyblue')
    axs[i].invert_yaxis()

plt.tight_layout()
plt.show()


#ax.set_title('Aug02 - 19388')

In [ ]:
#aggregated_rec= aggregated_rec.save(fodler="./sorting/recordingtest",progress_bar=True,verbose=True)
recording_chunk = recording_objects[0]
output_folder = "./sorting/CIRM_IMR90_org_16719_KS"
docker_image= "rohanmalige/benshalom:v3"
default_KS2_params = si.get_default_sorter_params('kilosort2')
print(default_KS2_params)
#default_KS2_params['keep_good_only'] = True
default_KS2_params['detect_threshold'] = 5.5
# default_KS2_params['projection_threshold']=[18, 10]
default_KS2_params['n_jobs'] = 32
# global_job_kwargs = dict(n_jobs=32, total_memory="8G", progress_bar=True)
# si.set_global_job_kwargs(**global_job_kwargs)
run_sorter = si.run_sorter('kilosort2',recording=recording_chunk, output_folder=output_folder,docker_image= docker_image,verbose=True,remove_existing_folder=True, **default_KS2_params)

## if running on NERSC:
#run_sorter_local("kilosort2",recording_chunk, output_folder="./sorting/FolicAcid10mg", delete_output_folder=False,verbose=True,with_output=True,**default_KS2_params)
#run_sorter = ss.run_sorter('kilosort2',recording= recording_chunk, output_folder="/mnt/disk15tb/mmpatil/Spikesorting/sorter_output/kilosort2",docker_image= True,verbose=True, **default_KS2_params)

In [ ]:
# loading the KS2 sorted object
sortingKS3 = run_sorter.remove_empty_units()
sortingKS3 = si.remove_excess_spikes(sortingKS3,recording_chunk) #Sometimes KS returns spikes outside the number of samples. < https://github.com/SpikeInterface/spikeinterface/pull/1378>

sortingKS3= sortingKS3.save(folder = output_folder+'2',overwrite=True)
#sorting_KS3 = s.Kilosort2Sorter._get_result_from_folder('./sorting/FolicAcidT2M07038_2_KS/sorter_output')
total_units = sortingKS3.get_unit_ids()
print(len(total_units))
#print(len(total_units))
channel_ids = recording_chunk.get_channel_ids()
print(channel_ids)

In [ ]:
waveform_folder = "./sorting/CIRM_IMR90_org_16719_WF"
job_kwargs = dict(n_jobs=32, chunk_duration="1s", progress_bar=True)
#waveforms = si.extract_waveforms(recording_chunk,sorting_KS3,folder="./waveformsblock1_7min",overwrite=True, ms_before=1., ms_after=2.,**job_kwargs)

#recording1.annotate(is_filtered=True)
waveforms = si.extract_waveforms(recording_chunk,sortingKS3,folder=waveform_folder,sparse=False,overwrite=True,**job_kwargs)
print(waveforms)

In [ ]:
#extract spiketimes
fig, ax1 = plt.subplots(figsize=(8,4))
spike_times = {}
fs = recording_chunk.get_sampling_frequency()
for idx, unit_id in enumerate(waveforms.unit_ids):
    #print(unit_id)
    spike_train = sortingKS3.get_unit_spike_train(unit_id,start_frame=0*fs,end_frame=120*fs)
    #print(spike_train)
    if len(spike_train) > 0:
        spike_times[idx] = spike_train / float(fs)
        #print(spike_times[unit_id])
       # print(unit_id*np.ones_like(spike_times[unit_id]))
        # ax1.plot(spike_times[idx],waveforms.channel_ids_to_indices([str(int(extremum_channels_ids[unit_id]))])*np.ones_like(spike_times[idx]),
        #                      marker='|', mew=1, markersize=3,
        #                      ls='',color='royalblue')
        ax1.plot(spike_times[idx],unit_id*np.ones_like(spike_times[idx]),
                             marker='|', mew=1, markersize=3,
                             ls='',color='royalblue')
        ax1.set_title('Raster plot')
        ax1.set_xlabel('time s')
        ax1.set_ylabel("channels")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve, find_peaks
from scipy.stats import norm
def plot_network_activity(ax,SpikeTimes, min_peak_distance=1.0, binSize=0.1, gaussianSigma=0.16, thresholdBurst=1.2, figSize=(10, 6)):
    relativeSpikeTimes = []
    units = 0
    for unit_id, spike_times in SpikeTimes.items():
        relativeSpikeTimes.extend(spike_times) 
        units += 1 # Set the first spike time to 0
    
    relativeSpikeTimes = np.array(relativeSpikeTimes)
    relativeSpikeTimes.sort() # Convert to NumPy array

    # Step 1: Bin all spike times into small time windows
    timeVector = np.arange(min(relativeSpikeTimes), max(relativeSpikeTimes), binSize)  # Time vector for binning
    binnedTimes, _ = np.histogram(relativeSpikeTimes, bins=timeVector)  # Bin spike times
    binnedTimes = np.append(binnedTimes, 0)  # Append 0 to match MATLAB's binnedTimes length

    # Step 2: Smooth the binned spike times with a Gaussian kernel
    kernelRange = np.arange(-3*gaussianSigma, 3*gaussianSigma + binSize, binSize)  # Range for Gaussian kernel
    kernel = norm.pdf(kernelRange, 0, gaussianSigma)  # Gaussian kernel
    kernel *= binSize  # Normalize kernel by bin size
    firingRate = convolve(binnedTimes, kernel, mode='same') / binSize  # Convolve and normalize by bin size
    firingRate = firingRate / units  # Convert to Hz

    # Create a new figure with a specified size (width, height)
    #fig, ax = plt.subplots(figsize=figSize)

    # Plot the smoothed network activity
    ax.plot(timeVector, firingRate, color='royalblue')
    # Restrict the plot to the first and last 100 ms
    ax.set_xlim([min(relativeSpikeTimes), max(relativeSpikeTimes)])
    ax.set_ylim([min(firingRate)*0.8, max(firingRate)*1.2])  # Set y-axis limits to min and max of firingRate
    ax.set_ylabel('Firing Rate [Hz]')
    ax.set_xlabel('Time [ms]')
    ax.set_title('Network Activity', fontsize=11)

    # Step 3: Peak detection on the smoothed firing rate curve
    rmsFiringRate = np.sqrt(np.mean(firingRate**2))  # Calculate RMS of the firing rate
    peaks, properties = find_peaks(firingRate, height=thresholdBurst * rmsFiringRate, distance=min_peak_distance)  # Find peaks above the threshold
    burstPeakTimes = timeVector[peaks]  # Convert peak indices to times
    burstPeakValues = properties['peak_heights']  # Get the peak values

    # Plot the threshold line and burst peaks
    ax.plot(np.arange(timeVector[-1]), thresholdBurst * rmsFiringRate * np.ones(np.ceil(timeVector[-1]).astype(int)), color='gray')
    ax.plot(burstPeakTimes, burstPeakValues, 'or')  # Plot burst peaks as red circles    

    return fig,ax


In [ ]:
# Parameters
window_size = 1.0  # seconds
step_size = 0.1    # seconds
threshold = 20 / window_size   # Hz

# Preparation for optimized plotting
spike_times_red = []  # Spikes exceeding threshold
spike_indices_red = []
spike_times_blue = []  # Spikes below threshold
spike_indices_blue = []

# Use vectorized operations for moving window analysis
for idx, spike_times_array in spike_times.items():
    end_times = np.arange(window_size, np.max(spike_times_array) + step_size, step_size)
    start_times = end_times - window_size
    
    # Compute the number of spikes in each window using vectorization
    for start_time, end_time in zip(start_times, end_times):
        spikes_in_window = spike_times_array[(spike_times_array >= start_time) & (spike_times_array < end_time)]
        spiking_rate = len(spikes_in_window) / window_size
        
        # Classify and collect spikes based on threshold
        if spiking_rate >= threshold:
            spike_times_red.extend(spikes_in_window)
            spike_indices_red.extend([idx] * len(spikes_in_window))
        else:
            spike_times_blue.extend(spikes_in_window)
            spike_indices_blue.extend([idx] * len(spikes_in_window))

# Plotting in a two-row subplot
fig, axs = plt.subplots(2, 1, figsize=(8, 8),sharex=True)
plt.rcParams.update({'font.size': 10})
plt.rcParams.update({'font.family': 'Arial'})
# Plot the raster plot with highlighted spikes below threshold
axs[0].scatter(spike_times_blue, spike_indices_blue, marker='|', color='cornflowerblue', rasterized=True,s=1)
axs[0].set_title('Raster Plot with Highlighted Spikes (Below Threshold)')
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('Unit ID')
# Plot the raster plot with highlighted spikes above threshold
axs[0].scatter(spike_times_red, spike_indices_red, marker='|', color='black', rasterized=True,s=1)
axs[0].set_title('Raster Plot with Highlighted Spikes (Above Threshold)')
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('Unit ID')
# Call the plot_network_activity function and pass the SpikeTimes dictionary
plot_network_activity(axs[1],spike_times, figSize=(8, 4),binSize=0.1, gaussianSigma=0.2,min_peak_distance=10, thresholdBurst=2)

plt.tight_layout()
plt.savefig('/home/mmp/Documents/CIRM_25MAR/oragnoid_16719/network_plot.svg', dpi=300, format='svg')
#plt.show()





## NOw sorting the block scans.



In [ ]:
local_path= '/mnt/disk20tb/Organoids/Maxone/IMR90_T3_03262024_AP/IMR90_T3_03262024_AP/Trace_20240328_15_57_19.raw.h5' #network data from chip 16848

def get_recording_objects(local_path):
    recording_objects = []
    
    for i in range(1):
        well_id = f'well{i:03d}'
        stream_id = f'well{i:03d}'

        recording = si.read_maxwell(local_path, stream_id=stream_id)
       
        
        channel_ids = recording.get_channel_ids()
        fs = recording.get_sampling_frequency()
        num_chan = recording.get_num_channels()
        num_seg = recording.get_num_segments()
        total_recording = recording.get_total_duration()

        print(f'Well ID: {well_id}')
        print('Sampling frequency:', fs)
        print('Number of channels:', num_chan)
        print('Number of segments:', num_seg)
        print(f"Total recording: {total_recording} s")
        print()
        recording_bp = si.bandpass_filter(recording, freq_min=300, freq_max=3000)

        recodring_cmr = si.common_reference(recording_bp, reference='global', operator='median')
        #recording_chunk = recodring_cmr.frame_slice(start_frame= 1*fs,end_frame=425*fs)
        recording_chunk = recodring_cmr.frame_slice(start_frame= 0*fs,end_frame=300*fs)
        #recording_chunk =si.scale(recording_chunk, gain=3.0)
        print(f"chunk duration: {recording_chunk.get_total_duration()} s")
        recording_objects.append(recording_chunk)
    return recording_objects

recording_objects = get_recording_objects(local_path)

In [ ]:
#aggregated_rec= aggregated_rec.save(fodler="./sorting/recordingtest",progress_bar=True,verbose=True)
recording_chunk = recording_objects[0]
output_folder = "./sorting/CIRM_org_16719_TraceBLOCK_KS"
docker_image= "rohanmalige/benshalom:v3"
default_KS2_params = si.get_default_sorter_params('kilosort2')
print(default_KS2_params)
#default_KS2_params['keep_good_only'] = True
default_KS2_params['detect_threshold'] = 5.5
# default_KS2_params['projection_threshold']=[18, 10]
default_KS2_params['n_jobs'] = 32
# global_job_kwargs = dict(n_jobs=32, total_memory="8G", progress_bar=True)
# si.set_global_job_kwargs(**global_job_kwargs)
run_sorter = si.run_sorter('kilosort2',recording=recording_chunk, output_folder=output_folder,docker_image= docker_image,verbose=True,remove_existing_folder=True, **default_KS2_params)

## if runni

In [ ]:
# loading the KS2 sorted object
sortingKS3 = run_sorter.remove_empty_units()
sortingKS3 = si.remove_excess_spikes(sortingKS3,recording_chunk) #Sometimes KS returns spikes outside the number of samples. < https://github.com/SpikeInterface/spikeinterface/pull/1378>

sortingKS3= sortingKS3.save(folder = output_folder+'2',overwrite=True)
#sorting_KS3 = s.Kilosort2Sorter._get_result_from_folder('./sorting/FolicAcidT2M07038_2_KS/sorter_output')
total_units = sortingKS3.get_unit_ids()
print(len(total_units))
#print(len(total_units))
channel_ids = recording_chunk.get_channel_ids()
print(channel_ids)

In [ ]:
waveform_folder = "./sorting/CIRM_16719_traceblock_WF"
job_kwargs = dict(n_jobs=32, chunk_duration="1s", progress_bar=True)
#waveforms = si.extract_waveforms(recording_chunk,sorting_KS3,folder="./waveformsblock1_7min",overwrite=True, ms_before=1., ms_after=2.,**job_kwargs)

#recording1.annotate(is_filtered=True)
waveforms = si.extract_waveforms(recording_chunk,sortingKS3,folder=waveform_folder,sparse=False,overwrite=True,**job_kwargs)
print(waveforms)

In [ ]:
%matplotlib widget
#waveforms=si.compute_unit_locations(waveforms)
waveforms=si.load_waveforms(waveform_folder)


In [ ]:
si.plot_unit_locations(waveforms,with_channel_ids=False,unit_ids=[10])

In [ ]:
si.plot_unit_templates(waveforms,unit_ids=[52])

In [ ]:
%matplotlib widget
objs = si.plot_unit_templates(waveforms,unit_ids=[52])

In [ ]:
objs.figure.savefig('/home/mmp/Documents/CIRM_25MAR/oragnoid_16719_traceblock/traceblock_unit52.svg', dpi=300, format='svg')

In [ ]:
locations = si.compute_unit_locations(waveforms)
unit_locations =dict(zip(waveforms.unit_ids,locations))
print(unit_locations)


In [ ]:
si.plot_unit_summary(waveforms,unit_id=42)

In [ ]:
extremum_channels_ids =si.get_template_extremum_channel(waveforms,peak_sign ='neg',mode='at_index')
for i, unit_id in enumerate(waveforms.unit_ids):
    fig, ax = plt.subplots()
    
    wf = waveforms.get_waveforms(unit_id)
    #print(wf.shape)
    #print(int(extremum_channels[unit_id]))
    channel_id_str = str(int(extremum_channels_ids[unit_id]))
    number = waveforms.channel_ids_to_indices([channel_id_str])
    #number =[539]
    #print(number)
    ax.plot(wf[:,:, number[0]].T,  lw=1, color='black', alpha=0.1, linestyle='-', marker='', markersize=0)
    ax.set_title(f"waveforms of a putative neuron")
    ax.set_ylabel("Amplitude (µV)")
    ax.set_xlabel("Sampled timepoints (5e-2 ms)")
    ax.set_facecolor('white')  # Set the background color to black

    # Customize the appearance of tick labels and axes
    ax.tick_params(axis='x', colors='black')
    ax.tick_params(axis='y', colors='black')
    # ax.spines['bottom'].set_color('white')
    ax.spines['top'].set_color('white')
    ax.spines['right'].set_color('white')
    # ax.spines['left'].set_color('white')
    
    plt.savefig(f'/home/mmp/Documents/CIRM_25MAR/oragnoid_16719_traceblock/waveforms/waveforms_unit_id_{unit_id}__channel_id_{channel_id_str}_channel_index_{number[0]}.svg', format='svg',dpi=300)
    plt.close(fig)

In [ ]:
extremum_channels_ids

In [ ]:
channel_ids = recording_chunk.get_channel_ids()

# Specify the particular channel ID you are interested in
specific_channel_id = '232' # Replace `desired_channel_id` with the actual ID you're looking for

index = np.where(np.array(channel_ids) == specific_channel_id)[0][0]
index

In [ ]:
traces = recording.get_traces(start_frame=172550, end_frame=188550, segment_index=0,return_scaled=True)
plt.figure(figsize=(22,2))                                                                                                                                                                                                                                                              
plt.plot(traces[:,232])

In [ ]:
traces = recording_chunk.get_traces(start_frame=493550, end_frame=501550, segment_index=0,return_scaled=True)
plt.figure(figsize=(22,2))                                                                                                                                                                                                                                                              
plt.plot(traces[:,304])
plt.savefig('/home/mmp/Documents/CIRM_25MAR/oragnoid_16719/trace_unit20.svg', format='svg',dpi=300)